# Empirical and theoretical probability
Counting from data versus counting the sample space, and watching the first approach the second as the trials grow.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## 1. Empirical probability from counts

In [ ]:
# 100 tosses gave 55 heads: empirical P(head) = count / trials
print('coin:', 55 / 100)

# 200 marble draws (with replacement) gave 80 red
print('red marble:', 80 / 200)

## 2. Empirical probability from a dataset
The Titanic data: drawing one passenger at random and reading `Pclass`.

In [ ]:
df = pd.read_csv('data/titanic_train.csv')
print(len(df), 'passengers')
# sample space: the values the column can take
print('sample space:', sorted(df['Pclass'].unique().tolist()))

# share of passengers in each class = empirical probability
print(df['Pclass'].value_counts(normalize=True).sort_index().round(3))

## 3. Theoretical probability
Favourable outcomes divided by all outcomes, when every outcome is equally likely.

In [ ]:
from fractions import Fraction

die = {1, 2, 3, 4, 5, 6}
less_than_3 = {x for x in die if x < 3}
# number of favourable outcomes / number of outcomes
print('P(3) =', Fraction(1, len(die)))
print('P(less than 3) =', Fraction(len(less_than_3), len(die)))

coins = ['HH', 'HT', 'TH', 'TT']
two_heads = [s for s in coins if s == 'HH']
print('P(two heads) =', Fraction(len(two_heads), len(coins)))

## 4. More trials, closer to the theory
The same coin, tossed 10 to 100,000 times (0 = tail, 1 = head).

In [ ]:
rng = np.random.default_rng(42)   # fixed seed: same tosses every run
for n in [10, 100, 1_000, 10_000, 100_000]:
    tosses = rng.integers(0, 2, n)   # 0 or 1
    # the mean of 0/1 values is the share of heads
    print(n, 'tosses: P(head) =', tosses.mean())

In [ ]:
# a die rolled 10, 1,000 and 100,000 times: share of each face
rng = np.random.default_rng(7)
for n in [10, 1_000, 100_000]:
    rolls = rng.integers(1, 7, n)            # 1 to 6
    shares = np.bincount(rolls, minlength=7)[1:] / n
    print(n, 'rolls:', shares.round(4))
print('theory: 1/6 =', round(1 / 6, 4))

### The running share of heads
After each toss, the share of heads so far. Three separate runs; the x axis is logarithmic.

In [ ]:
fig = go.Figure()
n = 100_000
steps = np.arange(1, n + 1)
for seed in [0, 1, 2]:
    tosses = np.random.default_rng(seed).integers(0, 2, n)
    running = tosses.cumsum() / steps     # share of heads after each toss
    # plot 400 log-spaced points only, to keep the notebook small
    keep = np.unique(np.logspace(0, 5, 400).astype(int)) - 1
    fig.add_scatter(x=steps[keep], y=running[keep], mode='lines', name=f'run {seed + 1}')
fig.add_hline(y=0.5, line_dash='dash')
fig.update_xaxes(type='log', title='number of tosses')
fig.update_yaxes(range=[0, 1], title='share of heads so far')
fig

## 5. The marble bag by simulation
20 red, 15 blue, 15 green; 200 draws with replacement.

In [ ]:
rng = np.random.default_rng(1)
bag = np.array(['red'] * 20 + ['blue'] * 15 + ['green'] * 15)
draws = rng.choice(bag, 200)   # with replacement
print(pd.Series(draws).value_counts() / 200)
print('theory: red 0.4, blue 0.3, green 0.3')

## 6. The addition rule, checked by counting

In [ ]:
A = {1, 3, 5}        # odd
B = {4, 5, 6}        # greater than 3
P = lambda E: Fraction(len(E), 6)
print('P(A or B) by counting:', P(A | B))
print('P(A) + P(B) - P(A and B):', P(A) + P(B) - P(A & B))
print('complement of A:', die - A, '->', 1 - P(A))